In [5]:
# https://colab.research.google.com/github/adin786/mot-data/blob/main/eda.ipynb#scrollTo=FWP7L3AXwsk9
# https://www.domo.com/learn/article/postgresql-for-data-analysis-a-complete-guide


csv/parquet into sqlite

In [6]:
import os
import re
import duckdb
from python_calamine import CalamineWorkbook

# Configuration paths
excel_path = "/home/user/py-pannguyen-learn/data/staging/21th/Used_Car_Sales_dataset.xlsx"
sqlite_path = "/home/user/py-pannguyen-learn/data/staging/21th/user_sale_data.db"
parquet_output_dir = "/home/user/py-pannguyen-learn/data/staging/21th/parquet"

# Ensure output directory for parquet files exists
os.makedirs(parquet_output_dir, exist_ok=True)

# 1. Discover all sheet names dynamically using Calamine
workbook = CalamineWorkbook.from_path(excel_path)
sheet_names = workbook.sheet_names

# 2. Initialize DuckDB and load required extensions
con = duckdb.connect()
con.execute("INSTALL excel; LOAD excel;")
con.execute("INSTALL sqlite; LOAD sqlite;")

# 3. Attach the target SQLite database
con.execute(f"ATTACH '{sqlite_path}' AS sqlite_db (TYPE SQLITE);")

# 4. Loop through each sheet, export to Parquet, and ingest into SQLite
for sheet in sheet_names:
    # Sanitize sheet name for safe SQL table identifiers
    safe_table_name = re.sub(r'\W+', '_', sheet).strip('_').lower()
    parquet_file = os.path.join(parquet_output_dir, f"{safe_table_name}.parquet")
    
    print(f"Processing sheet: '{sheet}' -> Table: '{safe_table_name}'")
    
    # Step A: Export sheet data directly to Parquet via DuckDB with header enabled
    con.execute(f"""
        COPY (SELECT * FROM read_xlsx('{excel_path}', sheet='{sheet}', header=true)) 
        TO '{parquet_file}' (FORMAT PARQUET);
    """)
    
    # Step B: Load the same sheet data straight into SQLite via DuckDB's SQLite extension with header enabled
    con.execute(f"""
        CREATE TABLE IF NOT EXISTS sqlite_db.{safe_table_name} AS 
        SELECT * FROM read_xlsx('{excel_path}', sheet='{sheet}', header=true);
    """)

# Clean up connection
con.close()
print("\nSuccessfully converted all sheets to Parquet files and loaded them into SQLite!")

Processing sheet: 'Sheet1' -> Table: 'sheet1'

Successfully converted all sheets to Parquet files and loaded them into SQLite!


In [7]:
# https://vishnutr.medium.com/exploratory-data-analysis-eda-using-sql-a-step-by-step-guide-1450ddac23b9
import duckdb
import sqlite3

con = duckdb.connect()
con.execute("INSTALL excel; LOAD excel;")
con.execute("INSTALL sqlite; LOAD sqlite;")

In [8]:
# 3. Attach your existing SQLite database file
con.execute("ATTACH '/home/user/py-pannguyen-learn/data/raw/sales_data_sample.db' AS sales_data_sample (TYPE SQLITE);")



In [9]:
# 4. Query the table and load the results directly into a Polars DataFrame
import polars as pl

sqlite_path = "/home/user/py-pannguyen-learn/data/raw/sales_data_sample.db"
con.execute(f"ATTACH '{sqlite_path}' AS sales_db (TYPE SQLITE);")



In [10]:
# 3. Query using the attached database alias and convert to Polars lazy frame
df = con.sql("SELECT * FROM sales_data_sample.sales_data;").pl(lazy=True)

# 4. Collect results and print row count
print(df.select(pl.len()).collect().item())

1000


In [11]:
df.collect_schema().names()

['Order_ID',
 'Date',
 'Category',
 'Region',
 'Customer_Segment',
 'Payment_Method',
 'Quantity',
 'Unit_Price',
 'Discount_Rate',
 'Customer_Rating',
 'Returned',
 'Subcategory',
 'Total_Price',
 'Discount_Amount',
 'Net_Sales',
 'Shipping_Cost',
 'Profit_Margin',
 'Profit',
 'Month']

In [12]:
con.sql("SHOW sales_data_sample.sales_data;")

column_nameVARCHAR,column_typeVARCHAR,nullVARCHAR,keyVARCHAR,defaultVARCHAR,extraVARCHAR
Order_ID,VARCHAR,YES,NULL,NULL,NULL
Date,VARCHAR,YES,NULL,NULL,NULL
Category,VARCHAR,YES,NULL,NULL,NULL
Region,VARCHAR,YES,NULL,NULL,NULL
Customer_Segment,VARCHAR,YES,NULL,NULL,NULL
Payment_Method,VARCHAR,YES,NULL,NULL,NULL
Quantity,DOUBLE,YES,NULL,NULL,NULL
Unit_Price,DOUBLE,YES,NULL,NULL,NULL
Discount_Rate,DOUBLE,YES,NULL,NULL,NULL
Customer_Rating,DOUBLE,YES,NULL,NULL,NULL


In [13]:
con.sql("DESCRIBE sales_data_sample.sales_data;")

column_nameVARCHAR,column_typeVARCHAR,nullVARCHAR,keyVARCHAR,defaultVARCHAR,extraVARCHAR
Order_ID,VARCHAR,YES,NULL,NULL,NULL
Date,VARCHAR,YES,NULL,NULL,NULL
Category,VARCHAR,YES,NULL,NULL,NULL
Region,VARCHAR,YES,NULL,NULL,NULL
Customer_Segment,VARCHAR,YES,NULL,NULL,NULL
Payment_Method,VARCHAR,YES,NULL,NULL,NULL
Quantity,DOUBLE,YES,NULL,NULL,NULL
Unit_Price,DOUBLE,YES,NULL,NULL,NULL
Discount_Rate,DOUBLE,YES,NULL,NULL,NULL
Customer_Rating,DOUBLE,YES,NULL,NULL,NULL


In [14]:
con.sql("PRAGMA table_info(sales_data_sample.sales_data);")

cidINTEGER,nameVARCHAR,typeVARCHAR,notnullBOOLEAN,dflt_valueVARCHAR,pkBOOLEAN
0,Order_ID,VARCHAR,False,NULL,False
1,Date,VARCHAR,False,NULL,False
2,Category,VARCHAR,False,NULL,False
3,Region,VARCHAR,False,NULL,False
4,Customer_Segment,VARCHAR,False,NULL,False
5,Payment_Method,VARCHAR,False,NULL,False
6,Quantity,DOUBLE,False,NULL,False
7,Unit_Price,DOUBLE,False,NULL,False
8,Discount_Rate,DOUBLE,False,NULL,False
9,Customer_Rating,DOUBLE,False,NULL,False


In [15]:
# count null
con.sql("SELECT COUNT(*) FROM sales_data_sample.sales_data")

count_star()BIGINT
1000


In [16]:
con.sql("SELECT * FROM sales_data_sample.sales_data LIMIT 1;")

Order_IDVARCHAR,DateVARCHAR,CategoryVARCHAR,RegionVARCHAR,Customer_SegmentVARCHAR,Payment_MethodVARCHAR,QuantityDOUBLE,Unit_PriceDOUBLE,Discount_RateDOUBLE,Customer_RatingDOUBLE,ReturnedBIGINT,SubcategoryVARCHAR,Total_PriceDOUBLE,Discount_AmountDOUBLE,Net_SalesDOUBLE,Shipping_CostDOUBLE,Profit_MarginDOUBLE,ProfitDOUBLE,MonthVARCHAR
ORD-10000,2024-07-12 00:00:00,Books,East,Business,PayPal,1.0,316.58,0.2,5.0,0,Comics,316.58,63.316,253.264,0.0,0.33,83.57712,2024-07


In [17]:
con.sql("""
    --Query counts missing values in specific columns
SELECT 
    COUNT(*) AS total_rows,
    SUM(CASE WHEN Order_ID IS NULL THEN 1 
        ELSE 0 END) AS missing_order_id,
    SUM(CASE WHEN Date IS NULL THEN 1 
        ELSE 0 END) AS missing_date
FROM sales_data_sample.sales_data;
        """)

total_rowsBIGINT,missing_order_idHUGEINT,missing_dateHUGEINT
1000,0,0


In [18]:
# 3. Retrieve all column names for the target table
columns_df = con.execute("DESCRIBE sales_data_sample.sales_data;").fetchdf()
columns_df

,column_name,column_type,null,key,default,extra
0,Order_ID,VARCHAR,YES,None,None,None
1,Date,VARCHAR,YES,None,None,None
2,Category,VARCHAR,YES,None,None,None
3,Region,VARCHAR,YES,None,None,None
4,Customer_Segment,VARCHAR,YES,None,None,None
5,Payment_Method,VARCHAR,YES,None,None,None
6,Quantity,DOUBLE,YES,None,None,None
7,Unit_Price,DOUBLE,YES,None,None,None
8,Discount_Rate,DOUBLE,YES,None,None,None
9,Customer_Rating,DOUBLE,YES,None,None,None


In [19]:


column_names = columns_df['column_name'].tolist()

# 4. Dynamically build a single SQL query to count nulls for all columns
# Using double quotes around column names protects against spaces or special characters
select_exprs = [
    f'COUNT(CASE WHEN "{col}" IS NULL THEN 1 END) AS "{col}_nulls"' 
    for col in column_names
]
query = f"SELECT {', '.join(select_exprs)} FROM sales_data_sample.sales_data;"

# 5. Execute and transpose the result for a clean, vertical checklist view
null_counts_df = con.execute(query).fetchdf().T
null_counts_df.columns = ["Null_Count"]

print("--- Null Counts per Column ---")
print(null_counts_df)

--- Null Counts per Column ---
                        Null_Count
Order_ID_nulls                   0
Date_nulls                       0
Category_nulls                   0
Region_nulls                     0
Customer_Segment_nulls           0
Payment_Method_nulls             0
Quantity_nulls                   0
Unit_Price_nulls                 0
Discount_Rate_nulls             51
Customer_Rating_nulls           55
Returned_nulls                   0
Subcategory_nulls                0
Total_Price_nulls                0
Discount_Amount_nulls            0
Net_Sales_nulls                  0
Shipping_Cost_nulls             42
Profit_Margin_nulls              0
Profit_nulls                     0
Month_nulls                      0


In [20]:
con.sql("""
        SELECT 
    MIN(Quantity) AS min_order_value,
    MAX(Quantity) AS max_order_value,
    AVG(Quantity) AS avg_order_value,
    SUM(Quantity) AS total_revenue
FROM sales_data_sample.sales_data;
        """)

min_order_valueDOUBLE,max_order_valueDOUBLE,avg_order_valueDOUBLE,total_revenueDOUBLE
1.0,9.0,5.078,5078.0


In [21]:
con.sql("""
        SELECT 
    MIN(Discount_Rate) AS min_order_value,
    MAX(Discount_Rate) AS max_order_value,
    AVG(Discount_Rate) AS avg_order_value,
    SUM(Discount_Rate) AS total_revenue
FROM sales_data_sample.sales_data;
        """)

min_order_valueDOUBLE,max_order_valueDOUBLE,avg_order_valueDOUBLE,total_revenueDOUBLE
0.0,0.2,0.09836670179135916,93.34999999999984


In [22]:
con.sql("""
        SELECT 
    MIN(Customer_Rating) AS min_order_value,
    MAX(Customer_Rating) AS max_order_value,
    AVG(Customer_Rating) AS avg_order_value,
    SUM(Customer_Rating) AS total_revenue
FROM sales_data_sample.sales_data;
        """)

min_order_valueDOUBLE,max_order_valueDOUBLE,avg_order_valueDOUBLE,total_revenueDOUBLE
1.0,5.0,3.708994708994709,3505.0


In [23]:
# --count distinct can be calculated for all the relevant dimensions
con.sql("SELECT COUNT(DISTINCT Region) AS unique_Region FROM sales_data_sample.sales_data;")

unique_RegionBIGINT
5


In [24]:
# --count distinct can be calculated for all the relevant dimensions
con.sql("SELECT COUNT(DISTINCT Category) AS unique_Category FROM sales_data_sample.sales_data;")

unique_CategoryBIGINT
5


In [25]:
# --count distinct can be calculated for all the relevant dimensions
con.sql("SELECT COUNT(DISTINCT Payment_Method) AS unique_Payment_Method FROM sales_data_sample.sales_data;")


unique_Payment_MethodBIGINT
5


In [26]:
# --display order statuses and their occurance 
# --in descending order of number of status occurance
con.sql("""SELECT Payment_Method, COUNT(*) AS Payment_Method_frequency
FROM sales_data_sample.sales_data
GROUP BY Payment_Method
ORDER BY Payment_Method_frequency DESC;
""")

Payment_MethodVARCHAR,Payment_Method_frequencyBIGINT
PayPal,217
Cash on Delivery,203
Credit Card,201
Debit Card,192
Bank Transfer,187
